In [18]:
# ============================================================
# ROCCHIO QUERY EXPANSION - CRANFIELD
# FINAL CODE - ONLY REQUIRED OUTPUT
# ===========================================================
import nltk
import ir_datasets
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
# ------------------------------------------------------------
# Download required NLTK resources
# ------------------------------------------------------------
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
# ------------------------------------------------------------
# Load Cranfield dataset
# ------------------------------------------------------------
dataset = ir_datasets.load("cranfield")
documents = [
    {'doc_id': doc.doc_id, 'text': doc.text}
    for doc in dataset.docs_iter()
]
queries = [
    {'query_id': query.query_id, 'text': query.text}
    for query in dataset.queries_iter()
]
# ------------------------------------------------------------
# Load relevance judgments
# ------------------------------------------------------------
relevance_judgments = {}
for qrel in dataset.qrels_iter():
    qid = qrel.query_id
    if qid not in relevance_judgments:
        relevance_judgments[qid] = {
            'relevant_docs': set(),
            'non_relevant_docs': set()
        }
    if qrel.relevance > 0:
        relevance_judgments[qid]['relevant_docs'].add(qrel.doc_id)
    else:
        relevance_judgments[qid]['non_relevant_docs'].add(qrel.doc_id)
# ------------------------------------------------------------
# Preprocessing
# ------------------------------------------------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    return " ".join(tokens)
for doc in documents:
    doc["processed_text"] = preprocess_text(doc["text"])
for query in queries:
    query["processed_text"] = preprocess_text(query["text"])
# ------------------------------------------------------------
# Create term-frequency vectors
# ------------------------------------------------------------
vectorizer = CountVectorizer()

all_texts = (
    [doc["processed_text"] for doc in documents]
    + [query["processed_text"] for query in queries]
)
vectorizer.fit(all_texts)
doc_tf_vectors = vectorizer.transform(
    [doc["processed_text"] for doc in documents]
)
query_tf_vectors = vectorizer.transform(
    [query["processed_text"] for query in queries]
)
feature_names = vectorizer.get_feature_names_out()
doc_id_to_idx = {
    doc["doc_id"]: i
    for i, doc in enumerate(documents)
}
query_id_to_idx = {
    query["query_id"]: i
    for i, query in enumerate(queries)
}
# ------------------------------------------------------------
# Rocchio - Ide Regular
# ------------------------------------------------------------
def rocchio_ide_regular(
    original_query,
    relevant_docs,
    alpha=1,
    beta=0.75
):
    modified = original_query.astype(float).copy()
    if len(relevant_docs) > 0:
        modified += beta * np.mean(relevant_docs, axis=0)
    return modified
# ------------------------------------------------------------
# Rocchio - Ide Dec Hi
# ------------------------------------------------------------
def rocchio_ide_dec_hi(
    original_query,
    relevant_docs,
    non_relevant_docs,
    alpha=1,
    beta=0.75,
    gamma=0.15
):
    modified = original_query.astype(float).copy()
    if len(relevant_docs) > 0:
        modified += beta * np.mean(relevant_docs, axis=0)
    if len(non_relevant_docs) > 0:
        modified -= gamma * non_relevant_docs[0].astype(float)
    return modified
# ------------------------------------------------------------
# Get top terms
# ------------------------------------------------------------
def get_top_terms(vector, n=5):
    scores = vector.flatten()
    indices = scores.argsort()[::-1]
    result = []
    for i in indices:
        if scores[i] > 0:
            result.append(
                (
                    feature_names[i],
                    round(float(scores[i]), 2)
                )
            )
        if len(result) == n:
            break
    return result
# ============================================================
# FINAL OUTPUT
# ============================================================
query_ids = ["1", "2", "3"]
print("Modified Query Terms Table:")
print()
print(
    f"{'keyword':<18}"
    f"{'Query':>7}"
    f"{'Dr':>5}"
    f"{'Dn':>5}"
    f"{'Rocchio':>10}"
    f"{'Ide_Regular':>14}"
    f"{'Ide_Dec_Hi':>12}"
)
print("-" * 75)
for qid in query_ids:
    qidx = query_id_to_idx[qid]
    original_query = query_tf_vectors[qidx].toarray().flatten()
    relevant_docs = [
        doc_tf_vectors[doc_id_to_idx[doc_id]]
        .toarray()
        .flatten()
        for doc_id in
        relevance_judgments
        .get(qid, {})
        .get("relevant_docs", [])
        if doc_id in doc_id_to_idx
    ]
    non_relevant_docs = [
        doc_tf_vectors[doc_id_to_idx[doc_id]]
        .toarray()
        .flatten()
        for doc_id in
        relevance_judgments
        .get(qid, {})
        .get("non_relevant_docs", [])
        if doc_id in doc_id_to_idx
    ]
    # Rocchio variants
    regular = rocchio_ide_regular(
        original_query,
        relevant_docs
    )
    dec_hi = rocchio_ide_dec_hi(
        original_query,
        relevant_docs,
        non_relevant_docs
    )
    # Collect important terms
    original_terms = get_top_terms(original_query, 5)
    regular_terms = get_top_terms(regular, 5)
    dec_hi_terms = get_top_terms(dec_hi, 5)
    all_terms = []
    for term, _ in original_terms:
        if term not in all_terms:
            all_terms.append(term)
    for term, _ in regular_terms:
        if term not in all_terms:
            all_terms.append(term)
    for term, _ in dec_hi_terms:
        if term not in all_terms:
            all_terms.append(term)
    # Print terms
    for term in all_terms[:10]:
        index = np.where(feature_names == term)[0][0]
        dr = (
            np.mean(
                [doc[index] for doc in relevant_docs]
            )
            if relevant_docs
            else 0
        )
        dn = (
            np.mean(
                [doc[index] for doc in non_relevant_docs]
            )
            if non_relevant_docs
            else 0
        )
        print(
            f"{term:<18}"
            f"{qid:>7}"
            f"{dr:>5.0f}"
            f"{dn:>5.0f}"
            f"{original_query[index]:>10.2f}"
            f"{regular[index]:>14.2f}"
            f"{dec_hi[index]:>12.2f}"
        )
# ============================================================
# MODIFIED QUERIES
# ============================================================
print()
print("Modified Queries:")
for qid in query_ids:
    qidx = query_id_to_idx[qid]
    original_query = query_tf_vectors[qidx].toarray().flatten()
    relevant_docs = [
        doc_tf_vectors[doc_id_to_idx[doc_id]]
        .toarray()
        .flatten()
        for doc_id in
        relevance_judgments
        .get(qid, {})
        .get("relevant_docs", [])
        if doc_id in doc_id_to_idx
    ]
    non_relevant_docs = [
        doc_tf_vectors[doc_id_to_idx[doc_id]]
        .toarray()
        .flatten()
        for doc_id in
        relevance_judgments
        .get(qid, {})
        .get("non_relevant_docs", [])
        if doc_id in doc_id_to_idx
    ]
    regular = rocchio_ide_regular(
        original_query,
        relevant_docs
    )
    dec_hi = rocchio_ide_dec_hi(
        original_query,
        relevant_docs,
        non_relevant_docs
    )
    regular_terms = [
        term
        for term, score in get_top_terms(regular, 10)
        if score > 0
    ]
    dec_hi_terms = [
        term
        for term, score in get_top_terms(dec_hi, 10)
        if score > 0
    ]
    print(
        f"Rocchio: "
        f"{' '.join(regular_terms)}"
    )
    print(
        f"Ide_Regular: "
        f"{' '.join(regular_terms)}"
    )
    print(
        f"Ide_Dec_Hi: "
        f"{' '.join(dec_hi_terms)}"
    )

Modified Query Terms Table:

keyword             Query   Dr   Dn   Rocchio   Ide_Regular  Ide_Dec_Hi
---------------------------------------------------------------------------
obeyed                  1    0    0      1.00          1.00        1.00
aircraft                1    1    0      1.00          1.48        1.48
heated                  1    0    0      1.00          1.19        1.19
speed                   1    0    1      1.00          1.32        1.17
aeroelastic             1    0    1      1.00          1.16        1.01
model                   1    1    4      1.00          1.70        1.10
similarity              1    0    4      1.00          1.32        0.72
high                    1    0    0      1.00          1.24        1.24
flight                  2    0    0      1.00          1.22        1.22
high                    2    0    0      1.00          1.25        1.25
problem                 2    1    3      1.00          1.53        1.08
aeroelastic             2    0 